# LGG Brain MRI Tumor Segmentation using U-Net
# Section 1 — Project Overview & Configuration

This project focuses on brain tumor segmentation in MRI scans using a U-Net convolutional neural network.

The complete pipeline includes dataset preparation, preprocessing, model training, quantitative evaluation, and qualitative visualization of segmentation results.

The model is evaluated using Dice Score, IoU, Precision, and Recall, with Dice Score used as the primary evaluation metric.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import json

In [ ]:
# Set random seeds for reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Select computation device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

In [ ]:
# Dataset paths
DATASET_DIR = r"..\data"

# Training configuration
NUM_EPOCHS = 30

print("Configuration loaded successfully.")

## 2. Dataset Exploration

This section explores the LGG brain MRI segmentation dataset, including its structure, image-mask pairs, dimensions, and sample images with corresponding segmentation masks.

In [ ]:
# Collect MRI images and corresponding masks

image_files = []
mask_files = []

for root, _, files in os.walk(DATASET_DIR):
    for file in files:
        if file.lower().endswith((".tif", ".tiff")):
            file_path = os.path.join(root, file)

            if file.lower().endswith("_mask.tif") or file.lower().endswith("_mask.tiff"):
                mask_files.append(file_path)
            else:
                image_files.append(file_path)

# Sort for deterministic ordering
image_files = sorted(image_files)
mask_files = sorted(mask_files)

# Match masks to images by filename, not by list position
mask_lookup = {
    os.path.splitext(os.path.basename(path))[0].replace("_mask", ""): path
    for path in mask_files
}


# Identify unmatched images/masks before enforcing strict pairing
image_names = {
    os.path.splitext(os.path.basename(p))[0] for p in image_files
}
mask_names = set(mask_lookup.keys())

missing_masks = image_names - mask_names   # images without a mask
missing_images = mask_names - image_names  # masks without an image

if missing_masks or missing_images:
    raise FileNotFoundError(
        f"Pairing failed. "
        f"Images without masks: {len(missing_masks)} | "
        f"Masks without images: {len(missing_images)}"
    )

paired_image_files = []
paired_mask_files = []

for image_path in image_files:
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    paired_image_files.append(image_path)
    paired_mask_files.append(mask_lookup[image_name])

image_files = paired_image_files
mask_files = paired_mask_files

print(f"Number of MRI images: {len(image_files)}")
print(f"Number of masks: {len(mask_files)}")
print(f"Matched image-mask pairs: {len(image_files)}")

In [ ]:
# Verify Image-Mask Matching

assert len(image_files) == len(mask_files), (
    "Number of images and masks do not match."
)

for image_path, mask_path in zip(image_files, mask_files):
    image_name = os.path.splitext(os.path.basename(image_path))[0]
    mask_name = os.path.splitext(os.path.basename(mask_path))[0].replace("_mask", "")

    assert image_name == mask_name, (
        f"Image-mask mismatch:\n{image_path}\n{mask_path}"
    )

print("All image-mask pairs are correctly matched.")

In [ ]:
# Check image and mask dimensions

sample_image_path = image_files[0]
sample_mask_path = mask_files[0]

sample_image = Image.open(sample_image_path)
sample_mask = Image.open(sample_mask_path)

print(f"Image size: {sample_image.size}")
print(f"Image mode: {sample_image.mode}")

print(f"Mask size: {sample_mask.size}")
print(f"Mask mode: {sample_mask.mode}")

In [ ]:
# Inspect unique pixel values in a sample mask

mask_array = np.array(sample_mask)

unique_values = np.unique(mask_array)

print("Unique mask values:", unique_values)

In [ ]:
empty_masks = 0
non_empty_masks = 0

for mask_path in mask_files:
    mask = np.array(Image.open(mask_path))

    if np.any(mask > 0):
        non_empty_masks += 1
    else:
        empty_masks += 1

print("Empty masks:", empty_masks)
print("Non-empty masks:", non_empty_masks)

In [ ]:
# Find one MRI with tumor and one MRI without tumor

tumor_index = next(
    i for i, path in enumerate(mask_files)
        if np.any(np.array(Image.open(path)) > 0)
        )

empty_index = next(
            i for i, path in enumerate(mask_files)
                if not np.any(np.array(Image.open(path)) > 0)
                )

tumor_image_path = image_files[tumor_index]
tumor_mask_path = mask_files[tumor_index]

empty_image_path = image_files[empty_index]
empty_mask_path = mask_files[empty_index]

print("Tumor sample:", tumor_image_path)
print("Empty-mask sample:", empty_image_path)

In [ ]:
print("Tumor mask unique values:",
      np.unique(np.array(Image.open(tumor_mask_path))))

print("Empty mask unique values:",
            np.unique(np.array(Image.open(empty_mask_path))))

## 3. Data Preprocessing

The MRI images and segmentation masks are preprocessed for U-Net training. MRI images are converted to grayscale and normalized, while the binary masks are converted from 0/255 to 0/1.

In [ ]:
# Load a sample image and mask

image = Image.open(tumor_image_path).convert("L")
mask = Image.open(tumor_mask_path).convert("L")

image_array = np.array(image, dtype=np.float32)
mask_array = np.array(mask, dtype=np.float32)

# Normalize MRI image to [0, 1]
image_array = image_array / 255.0

# Convert mask from [0, 255] to binary [0, 1]
mask_array = (mask_array > 0).astype(np.float32)

print("Image shape:", image_array.shape)
print("Image range:", image_array.min(), "to", image_array.max())

print("Mask shape:", mask_array.shape)
print("Mask values:", np.unique(mask_array))

In [ ]:
class MRISegmentationDataset(Dataset):
      def __init__(self, image_files, mask_files):
          self.image_files = image_files
          self.mask_files = mask_files

      def __len__(self):
          return len(self.image_files)

      def __getitem__(self, idx):
          image = Image.open(self.image_files[idx]).convert("L")
          mask = Image.open(self.mask_files[idx]).convert("L")

          image = np.array(image, dtype=np.float32) / 255.0
          mask = (np.array(mask, dtype=np.float32) > 0).astype(np.float32)

          image = torch.from_numpy(image).unsqueeze(0)
          mask = torch.from_numpy(mask).unsqueeze(0)

          return image, mask

In [ ]:
dataset = MRISegmentationDataset(image_files, mask_files)

image, mask = dataset[0]

print("Image shape:", image.shape)
print("Mask shape:", mask.shape)
print("Image range:", image.min().item(), "to", image.max().item())
print("Mask values:", torch.unique(mask))

In [ ]:
for i in range(len(dataset)):
    _, test_mask = dataset[i]

    if torch.any(test_mask > 0):
        print("First tumor sample index:", i)
        print("Mask values:", torch.unique(test_mask))
        break

## 4. Dataset Splitting

The image-mask pairs are split into training, validation, and test sets at the patient level to prevent data leakage between the subsets.

In [ ]:
# Extract patient IDs from image paths

patient_ids = [
    os.path.basename(os.path.dirname(path))
        for path in image_files
        ]

print("Total image-mask pairs:", len(image_files))
print("Unique patients:", len(set(patient_ids)))

In [ ]:
# Patient-Level Train / Validation / Test Split

unique_patients = sorted(set(patient_ids))

train_patients, temp_patients = train_test_split(
    unique_patients,
    test_size=0.30,
    random_state=SEED
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=SEED
)

print("Train patients:", len(train_patients))
print("Validation patients:", len(val_patients))
print("Test patients:", len(test_patients))

In [ ]:
# Assign image-mask pairs to train, validation, and test sets

train_images = []
train_masks = []

val_images = []
val_masks = []

test_images = []
test_masks = []

for image_path, mask_path, patient_id in zip(image_files, mask_files, patient_ids):

    if patient_id in train_patients:
        train_images.append(image_path)
        train_masks.append(mask_path)

    elif patient_id in val_patients:
        val_images.append(image_path)
        val_masks.append(mask_path)

    elif patient_id in test_patients:
        test_images.append(image_path)
        test_masks.append(mask_path)

print("Train pairs:", len(train_images))
print("Validation pairs:", len(val_images))
print("Test pairs:", len(test_images))

print("Total pairs:", len(train_images) + len(val_images) + len(test_images))

In [ ]:
# Verify that there is no patient overlap between splits

train_patient_set = set(train_patients)
val_patient_set = set(val_patients)
test_patient_set = set(test_patients)

print("Train ∩ Validation:", len(train_patient_set & val_patient_set))
print("Train ∩ Test:", len(train_patient_set & test_patient_set))
print("Validation ∩ Test:", len(val_patient_set & test_patient_set))

## 5. U-Net Model Architecture

A U-Net architecture is used for pixel-level brain tumor segmentation. The model consists of an encoder for feature extraction and a decoder for spatial reconstruction, with skip connections between corresponding levels.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        self.encoder1 = DoubleConv(in_channels, 64)
        self.encoder2 = DoubleConv(64, 128)
        self.encoder3 = DoubleConv(128, 256)
        self.encoder4 = DoubleConv(256, 512)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.bottleneck = DoubleConv(512, 1024)

        self.upconv4 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.decoder4 = DoubleConv(1024, 512)

        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.decoder3 = DoubleConv(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.decoder2 = DoubleConv(256, 128)

        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.decoder1 = DoubleConv(128, 64)

        self.output = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x):
        enc1 = self.encoder1(x)
        enc2 = self.encoder2(self.pool(enc1))
        enc3 = self.encoder3(self.pool(enc2))
        enc4 = self.encoder4(self.pool(enc3))

        bottleneck = self.bottleneck(self.pool(enc4))

        dec4 = self.upconv4(bottleneck)
        dec4 = torch.cat((dec4, enc4), dim=1)
        dec4 = self.decoder4(dec4)

        dec3 = self.upconv3(dec4)
        dec3 = torch.cat((dec3, enc3), dim=1)
        dec3 = self.decoder3(dec3)

        dec2 = self.upconv2(dec3)
        dec2 = torch.cat((dec2, enc2), dim=1)
        dec2 = self.decoder2(dec2)

        dec1 = self.upconv1(dec2)
        dec1 = torch.cat((dec1, enc1), dim=1)
        dec1 = self.decoder1(dec1)

        return self.output(dec1)

In [ ]:
model = UNet(in_channels=1, out_channels=1)

print(model)

In [ ]:
test_input = torch.randn(1, 1, 256, 256)

with torch.no_grad():
    test_output = model(test_input)

    print("Input shape:", test_input.shape)
    print("Output shape:", test_output.shape)

## 6. Loss Function & Evaluation Metrics

Binary Cross-Entropy (BCE) and Dice Loss are combined to optimize the segmentation model. Dice Score, IoU, Precision, and Recall are used to evaluate segmentation performance.

In [ ]:
def dice_coefficient(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()

    dice = (
        2.0 * intersection + smooth
    ) / (
        pred.sum() + target.sum() + smooth
    )

    return dice

In [ ]:
def iou_score(pred, target, threshold=0.5, smooth=1e-6):
    pred = (torch.sigmoid(pred) > threshold).float()

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection

    return (intersection + smooth) / (union + smooth)


def precision_score(pred, target, threshold=0.5, smooth=1e-6):
    pred = (torch.sigmoid(pred) > threshold).float()

    pred = pred.view(-1)
    target = target.view(-1)

    true_positive = (pred * target).sum()
    false_positive = (pred * (1 - target)).sum()

    return (true_positive + smooth) / (
        true_positive + false_positive + smooth
    )


def recall_score(pred, target, threshold=0.5, smooth=1e-6):
    pred = (torch.sigmoid(pred) > threshold).float()

    pred = pred.view(-1)
    target = target.view(-1)

    true_positive = (pred * target).sum()
    false_negative = ((1 - pred) * target).sum()

    return (true_positive + smooth) / (
        true_positive + false_negative + smooth
    )

In [ ]:
#=========================
#Pre-training Sanity Check
#=========================
#This cell checks tensor/model compatibility only.
#It is NOT used as a performance result.

test_image, test_mask = dataset[1]

test_image = test_image.unsqueeze(0)

with torch.no_grad():
    test_prediction = model(test_image)

    print("Dice:", dice_coefficient(test_prediction, test_mask.unsqueeze(0)).item())
    print("IoU:", iou_score(test_prediction, test_mask.unsqueeze(0)).item())
    print("Precision:", precision_score(test_prediction, test_mask.unsqueeze(0)).item())
    print("Recall:", recall_score(test_prediction, test_mask.unsqueeze(0)).item())

## 7. Model Training

The U-Net model is trained using the combined BCE and Dice loss. The Adam optimizer is used, and validation performance is monitored during training to select the best-performing model.

In [ ]:
train_dataset = MRISegmentationDataset(train_images, train_masks)
val_dataset = MRISegmentationDataset(val_images, val_masks)
test_dataset = MRISegmentationDataset(test_images, test_masks)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
batch_size = 16

# =========================
# DataLoaders
# =========================

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# =========================
# Device, Model, Optimizer
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)

learning_rate = 1e-4

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate
)

print("Device:", device)
print("Learning rate:", learning_rate)

In [ ]:
best_val_dice = 0.0

print("Epochs:", NUM_EPOCHS)
print("Best validation Dice:", best_val_dice)

In [ ]:
# =========================
# Training with Resume,
# Checkpoint, Early Stopping
# and Training History
# =========================

# =========================
# Training Settings
# =========================
criterion_bce = torch.nn.BCEWithLogitsLoss()

best_val_dice = 0.0
patience = 7
patience_counter = 0
start_epoch = 0

checkpoint_path = "checkpoint.pth"
best_model_path = "best_model.pth"


# =========================
# Training History
# =========================

history = {
    "train_loss": [],
    "val_loss": [],
    "train_dice": [],
    "val_dice": []
}


# =========================
# Resume from Checkpoint
# =========================

if os.path.exists(checkpoint_path):

    print("=" * 70)
    print("Checkpoint found. Resuming training...")
    print("=" * 70)

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    start_epoch = checkpoint["epoch"]
    best_val_dice = checkpoint["best_val_dice"]
    patience_counter = checkpoint["patience_counter"]

    # Restore training history if available
    if "history" in checkpoint:

        history = checkpoint["history"]

    print(f"Resuming from Epoch: {start_epoch}")
    print(f"Best Validation Dice: {best_val_dice:.4f}")
    print(
        f"Patience Counter: "
        f"{patience_counter}/{patience}"
    )

else:

    print("=" * 70)
    print("No checkpoint found.")
    print("Starting training from Epoch 1.")
    print("=" * 70)


print(f"Device: {device}")
print(f"Total Epochs: {NUM_EPOCHS}")
print(
    f"Learning Rate: "
    f"{optimizer.param_groups[0]['lr']}"
)
print(f"Checkpoint: {checkpoint_path}")
print(f"Best Model: {best_model_path}")
print("=" * 70)


# =========================
# Training Loop
# =========================

for epoch in range(start_epoch, NUM_EPOCHS):

    current_epoch = epoch + 1

    # -------------------------
    # Training
    # -------------------------

    model.train()

    train_loss = 0.0
    train_dice = 0.0

    for images, masks in train_loader:

        images = images.to(device)
        masks = masks.to(device).float()

        optimizer.zero_grad()

        # Raw model output (logits)
        outputs = model(images)

        # BCE + Dice Loss
        bce_loss = criterion_bce(
            outputs,
            masks
        )

        dice_loss = 1 - dice_coefficient(
            outputs,
            masks
        )

        loss = bce_loss + dice_loss

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        train_dice += dice_coefficient(
            outputs.detach(),
            masks
        ).item()

    train_loss /= len(train_loader)
    train_dice /= len(train_loader)


    # -------------------------
    # Validation
    # -------------------------

    model.eval()

    val_loss = 0.0
    val_dice = 0.0

    with torch.no_grad():

        for images, masks in val_loader:

            images = images.to(device)
            masks = masks.to(device).float()

            # Raw model output (logits)
            outputs = model(images)

            # BCE + Dice Loss
            bce_loss = criterion_bce(
                outputs,
                masks
            )

            dice_loss = 1 - dice_coefficient(
                outputs,
                masks
            )

            loss = bce_loss + dice_loss

            val_loss += loss.item()

            val_dice += dice_coefficient(
                outputs,
                masks
            ).item()

    val_loss /= len(val_loader)
    val_dice /= len(val_loader)


    # =========================
    # Save History
    # =========================

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_dice"].append(train_dice)
    history["val_dice"].append(val_dice)


    # =========================
    # Print Epoch Results
    # =========================

    print("\n" + "-" * 70)

    print(
        f"Epoch [{current_epoch:02d}/{NUM_EPOCHS}] "
        f"| Train Loss: {train_loss:.4f} "
        f"| Train Dice: {train_dice:.4f} "
        f"| Val Loss: {val_loss:.4f} "
        f"| Val Dice: {val_dice:.4f}"
    )


    # =========================
    # Save Best Model
    # =========================

    if val_dice > best_val_dice:

        best_val_dice = val_dice
        patience_counter = 0

        torch.save(
            model.state_dict(),
            best_model_path
        )

        print(
            f"★ NEW BEST MODEL"
            f" | Val Dice: {best_val_dice:.4f}"
            f" | Saved: {best_model_path}"
        )

    else:

        patience_counter += 1

        print(
            f"No improvement"
            f" | Patience: "
            f"{patience_counter}/{patience}"
        )


    # =========================
    # Save Checkpoint
    # =========================

    checkpoint = {
        "epoch": current_epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_dice": best_val_dice,
        "patience_counter": patience_counter,
        "history": history
    }

    torch.save(
        checkpoint,
        checkpoint_path
    )

    print(
        f"✓ Checkpoint saved"
        f" | Epoch: {current_epoch}"
        f" | Best Val Dice: "
        f"{best_val_dice:.4f}"
    )


    # =========================
    # Early Stopping
    # =========================

    if patience_counter >= patience:

        print("\n" + "=" * 70)
        print("EARLY STOPPING TRIGGERED")
        print(
            f"Stopped at Epoch: "
            f"{current_epoch}"
        )
        print(
            f"Best Validation Dice: "
            f"{best_val_dice:.4f}"
        )
        print(
            f"Best Model: "
            f"{best_model_path}"
        )
        print(
            f"Last Checkpoint: "
            f"{checkpoint_path}"
        )
        print("=" * 70)

        break


# =========================
# Training Completed
# =========================

print("\n" + "=" * 70)
print("TRAINING COMPLETED")
print(f"Last Epoch: {current_epoch}")
print(
    f"Best Validation Dice: "
    f"{best_val_dice:.4f}"
)
print(f"Best Model: {best_model_path}")
print(f"Checkpoint: {checkpoint_path}")
print("=" * 70)

In [ ]:
# =============================
# Get Training History
# =============================

train_losses = history["train_loss"]
val_losses = history["val_loss"]
train_dice_scores = history["train_dice"]
val_dice_scores = history["val_dice"]

epochs = list(range(1, len(train_losses) + 1))


# =============================
# Find Best Epoch
# =============================

best_epoch = val_dice_scores.index(
    max(val_dice_scores)
) + 1

best_dice = max(val_dice_scores)


# =============================
# Create Results DataFrame
# =============================

results_df = pd.DataFrame({
    "Epoch": epochs,
    "Train_Loss": train_losses,
    "Train_Dice": train_dice_scores,
    "Val_Loss": val_losses,
    "Val_Dice": val_dice_scores
})


# =============================
# Save Training Results
# =============================

results_df.to_csv(
    "training_results.csv",
    index=False
)


# =============================
# 1. Loss Curve
# =============================

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    train_losses,
    label="Train Loss"
)

plt.plot(
    epochs,
    val_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.savefig(
    "V1_loss_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()


# =============================
# 2. Validation Dice Curve
# =============================

plt.figure(figsize=(8, 5))

plt.plot(
    epochs,
    val_dice_scores,
    label="Validation Dice"
)

plt.axvline(
    best_epoch,
    linestyle="--",
    label=f"Best Epoch = {best_epoch}"
)

plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Validation Dice Score")
plt.legend()
plt.grid(True)

plt.savefig(
    "V1_validation_dice.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()


# =============================
# Save Best Results
# =============================

best_results = {
    "Best_Epoch": best_epoch,
    "Best_Validation_Dice": best_dice,
    "Train_Loss_at_Best_Epoch": train_losses[best_epoch - 1],
    "Train_Dice_at_Best_Epoch": train_dice_scores[best_epoch - 1],
    "Validation_Loss_at_Best_Epoch": val_losses[best_epoch - 1]
}

best_results_df = pd.DataFrame([best_results])

best_results_df.to_csv(
    "best_training_results.csv",
    index=False
)


# =============================
# Final Status
# =============================

print("Training results saved successfully.")
print("Saved files:")
print("- training_results.csv")
print("- best_training_results.csv")
print("- V1_loss_curve.png")
print("- V1_validation_dice.png")

## 8. Training Results & Analysis

Training and validation losses, together with validation Dice scores, are analyzed to monitor model convergence and identify potential overfitting during training.

In [ ]:
# =========================
# Plot Training Results
# =========================

epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs, train_losses, label="Training Loss")
plt.plot(epochs, val_losses, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(epochs, val_dice_scores, label="Validation Dice")

plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Validation Dice Score")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
best_epoch = val_dice_scores.index(max(val_dice_scores)) + 1
best_dice = max(val_dice_scores)

print("Best epoch:", best_epoch)
print("Best validation Dice:", best_dice)

## 9. Model Evaluation

The best-performing U-Net model is evaluated on the held-out test set using Dice Score, IoU, Precision, and Recall.

In [ ]:
best_model = UNet(in_channels=1, out_channels=1)

best_model.load_state_dict(
    torch.load(best_model_path, map_location=device)
)

best_model = best_model.to(device)
best_model.eval()

print("Best model loaded successfully.")

In [ ]:
# =========================
# Final Test Evaluation
# =========================

best_model.eval()

# Accumulators over the full test set
soft_intersection = 0.0
soft_pred_sum = 0.0
target_sum = 0.0

true_positive = 0.0
false_positive = 0.0
false_negative = 0.0

with torch.no_grad():
    for images, masks in test_loader:

        images = images.to(device)
        masks = masks.to(device).float()

        # Model output: logits
        logits = best_model(images)

        # Convert logits to probabilities
        probs = torch.sigmoid(logits)

        # Soft Dice components
        soft_intersection += (probs * masks).sum().item()
        soft_pred_sum += probs.sum().item()
        target_sum += masks.sum().item()

        # Binary prediction for IoU/Precision/Recall
        predictions = (probs > 0.5).float()

        # Confusion components
        true_positive += (predictions * masks).sum().item()
        false_positive += (predictions * (1 - masks)).sum().item()
        false_negative += ((1 - predictions) * masks).sum().item()

# =========================
# Calculate Final Metrics
# =========================

smooth = 1e-6

test_dice = (2.0 * soft_intersection + smooth) / (soft_pred_sum + target_sum + smooth)

test_iou = (true_positive + smooth) / (
    true_positive + false_positive + false_negative + smooth
)

test_precision = (true_positive + smooth) / (
    true_positive + false_positive + smooth
)

test_recall = (true_positive + smooth) / (
    true_positive + false_negative + smooth
)

print(f"Test Dice: {test_dice:.4f}")
print(f"Test IoU: {test_iou:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")

## 10. Qualitative Results

Representative test samples are visualized by comparing the original MRI image, the ground-truth tumor mask, and the predicted segmentation.

In [ ]:
# =========================
# Qualitative Results: 3 Test Samples
# =========================

sample_indices = [1, 12, 22]

fig, axes = plt.subplots(
    len(sample_indices),
    3,
    figsize=(12, 12)
)

with torch.no_grad():

    for row, sample_index in enumerate(sample_indices):

        image, mask = test_dataset[sample_index]

        input_tensor = image.unsqueeze(0).to(device)

        prediction = best_model(input_tensor)
        prediction = torch.sigmoid(prediction)
        prediction = (prediction > 0.5).float()

        image_np = image.squeeze().cpu().numpy()
        mask_np = mask.squeeze().cpu().numpy()
        prediction_np = prediction.squeeze().cpu().numpy()

        axes[row, 0].imshow(image_np, cmap="gray")
        axes[row, 0].set_title(f"MRI - Sample {sample_index}")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask_np, cmap="gray")
        axes[row, 1].set_title("Ground Truth")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(prediction_np, cmap="gray")
        axes[row, 2].set_title("Prediction")
        axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

## 11. Final Summary

The U-Net model is developed for brain tumor segmentation from LGG MRI scans. The complete pipeline includes patient-level dataset splitting, grayscale conversion, image normalization, binary mask preparation, model training with BCE and Dice loss, and quantitative and qualitative evaluation.

In [ ]:
print("Final Test Performance")
print("-" * 30)
print(f"Dice Score : {test_dice:.4f}")
print(f"IoU        : {test_iou:.4f}")
print(f"Precision  : {test_precision:.4f}")
print(f"Recall     : {test_recall:.4f}")

In [ ]:
# Save previous training results
previous_results = {
    "best_epoch": int(torch.tensor(val_dice_scores).argmax().item() + 1),
    "best_val_dice": float(max(val_dice_scores)),
    "last_epoch": len(train_losses),
    "last_train_loss": float(train_losses[-1]),
    "last_val_loss": float(val_losses[-1]),
    "last_val_dice": float(val_dice_scores[-1]),
    "train_losses": [float(x) for x in train_losses],
    "val_losses": [float(x) for x in val_losses],
    "val_dice_scores": [float(x) for x in val_dice_scores],

    "test_dice": float(test_dice),
    "test_iou": float(test_iou),
    "test_precision": float(test_precision),
    "test_recall": float(test_recall)
}

# Save as JSON
with open("V1_training_results.json", "w") as f:
    json.dump(previous_results, f, indent=4)

print("V1 training results saved successfully!")
print(f"Best Epoch: {previous_results['best_epoch']}")
print(f"Best Val Dice: {previous_results['best_val_dice']:.4f}")
print(f"Last Epoch: {previous_results['last_epoch']}")
print(f"Last Train Loss: {previous_results['last_train_loss']:.4f}")
print(f"Last Val Loss: {previous_results['last_val_loss']:.4f}")
print(f"Last Val Dice: {previous_results['last_val_dice']:.4f}")
print(f"Test Dice: {previous_results['test_dice']:.4f}")
print(f"Test IoU: {previous_results['test_iou']:.4f}")
print(f"Test Precision: {previous_results['test_precision']:.4f}")
print(f"Test Recall: {previous_results['test_recall']:.4f}")